In [ ]:
import pandas as pd
from pathlib import Path

file = Path(
    "../data/raw/primary/"
    "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv"
)

inpatient = pd.read_csv(file)

print("Shape:", inpatient.shape)
print(inpatient.head())
print(inpatient.columns.tolist())

Shape: (66773, 81)
        DESYNPUF_ID           CLM_ID  SEGMENT  CLM_FROM_DT  CLM_THRU_DT  \
0  00013D2EFD8E45D1  196661176988405        1   20100312.0   20100313.0   
1  00016F745862898F  196201177000368        1   20090412.0   20090418.0   
2  00016F745862898F  196661177015632        1   20090831.0   20090902.0   
3  00016F745862898F  196091176981058        1   20090917.0   20090920.0   
4  00016F745862898F  196261176983265        1   20100626.0   20100701.0   

  PRVDR_NUM  CLM_PMT_AMT  NCH_PRMRY_PYR_CLM_PD_AMT  AT_PHYSN_NPI  \
0    2600GD       4000.0                       0.0  3.139084e+09   
1    3900MB      26000.0                       0.0  6.476809e+09   
2    3900HM       5000.0                       0.0  6.119985e+08   
3    3913XU       5000.0                       0.0  4.971603e+09   
4    3900MB      16000.0                       0.0  6.408400e+09   

   OP_PHYSN_NPI  ...  HCPCS_CD_36  HCPCS_CD_37 HCPCS_CD_38  HCPCS_CD_39  \
0           NaN  ...          NaN          NaN

In [3]:
print("Rows:", inpatient.shape[0])
print("Columns:", inpatient.shape[1])

Rows: 66773
Columns: 81


In [6]:
print(
    "Missing DESYNPUF_ID:",
    inpatient["DESYNPUF_ID"].isna().sum()
)

print(
    "Missing CLM_ID:",
    inpatient["CLM_ID"].isna().sum()
)

print(
    "Unique beneficiaries:",
    inpatient["DESYNPUF_ID"].nunique()
)

print(
    "Unique claims:",
    inpatient["CLM_ID"].nunique()
)

Missing DESYNPUF_ID: 0
Missing CLM_ID: 0
Unique beneficiaries: 37780
Unique claims: 66705


In [7]:
print(
    "Duplicate CLM_ID:",
    inpatient["CLM_ID"].duplicated().sum()
)

Duplicate CLM_ID: 68


In [8]:
print(
    inpatient["SEGMENT"].value_counts(dropna=False)
)
segment_counts = (
    inpatient
    .groupby("CLM_ID")
    .size()
)

print(segment_counts.value_counts().sort_index())

SEGMENT
1    66705
2       68
Name: count, dtype: int64
1    66637
2       68
Name: count, dtype: int64


In [9]:
date_columns = [
    "CLM_FROM_DT",
    "CLM_THRU_DT",
    "CLM_ADMSN_DT",
    "NCH_BENE_DSCHRG_DT"
]

for col in date_columns:
    inpatient[col] = pd.to_datetime(
        inpatient[col],
        format="%Y%m%d",
        errors="coerce"
    )
print(inpatient[date_columns].dtypes)

CLM_FROM_DT           datetime64[ns]
CLM_THRU_DT           datetime64[ns]
CLM_ADMSN_DT          datetime64[ns]
NCH_BENE_DSCHRG_DT    datetime64[ns]
dtype: object


In [10]:
for col in date_columns:
    print(
        col,
        "→",
        inpatient[col].min(),
        "to",
        inpatient[col].max()
    )

CLM_FROM_DT → 2007-11-27 00:00:00 to 2010-12-30 00:00:00
CLM_THRU_DT → 2008-01-01 00:00:00 to 2010-12-31 00:00:00
CLM_ADMSN_DT → 2007-11-27 00:00:00 to 2010-12-30 00:00:00
NCH_BENE_DSCHRG_DT → 2008-01-01 00:00:00 to 2010-12-31 00:00:00


In [12]:
invalid_claim_dates = (
    inpatient["CLM_FROM_DT"].notna()
    &
    inpatient["CLM_THRU_DT"].notna()
    &
    (
        inpatient["CLM_FROM_DT"]
        > inpatient["CLM_THRU_DT"]
    )
)

print(
    "CLM_FROM_DT after CLM_THRU_DT:",
    invalid_claim_dates.sum()
)
invalid_admission_dates = (
    inpatient["CLM_ADMSN_DT"].notna()
    &
    inpatient["NCH_BENE_DSCHRG_DT"].notna()
    &
    (
        inpatient["CLM_ADMSN_DT"]
        > inpatient["NCH_BENE_DSCHRG_DT"]
    )
)

print(
    "Admission after discharge:",
    invalid_admission_dates.sum()
)

CLM_FROM_DT after CLM_THRU_DT: 0
Admission after discharge: 0


In [13]:
financial_cols = [
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "NCH_BENE_IP_DDCTBL_AMT",
    "NCH_BENE_PTA_COINSRNC_LBLTY_AM",
    "NCH_BENE_BLOOD_DDCTBL_LBLTY_AM",
    "CLM_PASS_THRU_PER_DIEM_AMT"
]

print(
    inpatient[financial_cols]
    .describe()
    .T
)

                                  count         mean          std     min  \
CLM_PMT_AMT                     66773.0  9573.632756  9315.073232 -8000.0   
NCH_PRMRY_PYR_CLM_PD_AMT        66773.0   398.899256  3663.463023     0.0   
NCH_BENE_IP_DDCTBL_AMT          64595.0  1057.058844    29.650916  1024.0   
NCH_BENE_PTA_COINSRNC_LBLTY_AM  66773.0    90.028904  1033.615068     0.0   
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM  66773.0     1.590313    40.163847     0.0   
CLM_PASS_THRU_PER_DIEM_AMT      66773.0    28.979228    75.606458     0.0   

                                   25%     50%      75%      max  
CLM_PMT_AMT                     4000.0  7000.0  11000.0  57000.0  
NCH_PRMRY_PYR_CLM_PD_AMT           0.0     0.0      0.0  68000.0  
NCH_BENE_IP_DDCTBL_AMT          1024.0  1068.0   1068.0   1100.0  
NCH_BENE_PTA_COINSRNC_LBLTY_AM     0.0     0.0      0.0  34000.0  
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM     0.0     0.0      0.0   2000.0  
CLM_PASS_THRU_PER_DIEM_AMT         0.0     0.0     10.0   

In [14]:
print(
    inpatient[financial_cols]
    .isna()
    .sum()
)

CLM_PMT_AMT                          0
NCH_PRMRY_PYR_CLM_PD_AMT             0
NCH_BENE_IP_DDCTBL_AMT            2178
NCH_BENE_PTA_COINSRNC_LBLTY_AM       0
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM       0
CLM_PASS_THRU_PER_DIEM_AMT           0
dtype: int64


In [15]:
missing_report = pd.DataFrame({
    "missing_count": inpatient.isna().sum(),
    "missing_percent": (
        inpatient.isna().mean() * 100
    ).round(2)
})

print(
    missing_report
    .sort_values("missing_percent", ascending=False)
    .to_string()
)

                                missing_count  missing_percent
HCPCS_CD_5                              66773           100.00
HCPCS_CD_6                              66773           100.00
HCPCS_CD_24                             66773           100.00
HCPCS_CD_23                             66773           100.00
HCPCS_CD_22                             66773           100.00
HCPCS_CD_21                             66773           100.00
HCPCS_CD_20                             66773           100.00
HCPCS_CD_19                             66773           100.00
HCPCS_CD_18                             66773           100.00
HCPCS_CD_17                             66773           100.00
HCPCS_CD_16                             66773           100.00
HCPCS_CD_15                             66773           100.00
HCPCS_CD_14                             66773           100.00
HCPCS_CD_13                             66773           100.00
HCPCS_CD_12                             66773          

In [16]:
print(
    "Missing providers:",
    inpatient["PRVDR_NUM"].isna().sum()
)

print(
    "Unique providers:",
    inpatient["PRVDR_NUM"].nunique()
)

print(
    inpatient["PRVDR_NUM"]
    .value_counts()
    .head(20)
)

Missing providers: 0
Unique providers: 2675
PRVDR_NUM
23006G    772
1000AH    634
3600NG    537
1401RR    431
2600ZT    422
2100WC    400
3400XN    396
0501MA    359
4500DP    337
1002DC    325
3100JN    315
1100TC    303
4400MM    301
0400AC    299
1700JJ    297
0503SD    296
3301QB    287
2201YM    284
3401MN    268
3100VC    261
Name: count, dtype: int64


In [17]:
print(
    inpatient["CLM_UTLZTN_DAY_CNT"]
    .describe()
)
print(
    "Negative utilization days:",
    (
        inpatient["CLM_UTLZTN_DAY_CNT"] < 0
    ).sum()
)

count    66705.000000
mean         5.582895
std          6.284463
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max        136.000000
Name: CLM_UTLZTN_DAY_CNT, dtype: float64
Negative utilization days: 0


In [18]:
diagnosis_cols = [
    col for col in inpatient.columns
    if "ICD9_DGNS_CD" in col
]

procedure_cols = [
    col for col in inpatient.columns
    if "ICD9_PRCDR_CD" in col
]

hcpcs_cols = [
    col for col in inpatient.columns
    if "HCPCS_CD" in col
]

print("Diagnosis columns:", len(diagnosis_cols))
print("Procedure columns:", len(procedure_cols))
print("HCPCS columns:", len(hcpcs_cols))

print(
    "Diagnosis columns:",
    diagnosis_cols
)

print(
    "Procedure columns:",
    procedure_cols
)

Diagnosis columns: 11
Procedure columns: 6
HCPCS columns: 45
Diagnosis columns: ['ADMTNG_ICD9_DGNS_CD', 'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4', 'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8', 'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10']
Procedure columns: ['ICD9_PRCDR_CD_1', 'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4', 'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6']


In [19]:
inpatient["CLAIM_KEY"] = (
    inpatient["CLM_ID"].astype(str)
    + "_"
    + inpatient["SEGMENT"].astype(str)
)

print(
    "Unique CLAIM_KEY:",
    inpatient["CLAIM_KEY"].nunique()
)

print(
    "Rows:",
    len(inpatient)
)

Unique CLAIM_KEY: 66773
Rows: 66773


In [20]:
segment_2 = inpatient[
    inpatient["SEGMENT"] == 2
]

print(segment_2.shape)

print(
    segment_2[
        [
            "DESYNPUF_ID",
            "CLM_ID",
            "SEGMENT",
            "PRVDR_NUM",
            "CLM_FROM_DT",
            "CLM_THRU_DT",
            "CLM_PMT_AMT",
            "CLM_UTLZTN_DAY_CNT"
        ]
    ].head(20)
)

(68, 82)
            DESYNPUF_ID           CLM_ID  SEGMENT PRVDR_NUM CLM_FROM_DT  \
1775   064E6D7106A6AB4B  196901176966106        2    3400ZQ         NaT   
2579   094DCF21CB0EBFE7  196531176970382        2    1701MN         NaT   
4068   0EEB3620D7B54C24  196821176998887        2    0300GU         NaT   
5703   1537674E8106621A  196101177023665        2    3600CC         NaT   
6068   169529AFAE961F6D  196231176993156        2    1101AT         NaT   
6556   18595F4AED7F919A  196251177002479        2    3230SM         NaT   
7195   1A9C572F75D2633F  196241176973103        2    0700WA         NaT   
7398   1B70055EC5EE77DC  196041177011944        2    0502VR         NaT   
7857   1D3336365D8BAF3A  196861177018065        2    39008N         NaT   
8001   1DBE1A52D496F962  196861177005835        2    3901VU         NaT   
9431   230D81C8F2AC25F1  196851176990818        2    1002DC         NaT   
9845   249ACA7B8B5FE0A3  196801177000710        2    4400MM         NaT   
10048  2570A19C4

In [21]:
segment_2_ids = set(
    segment_2["CLM_ID"]
)

segment_1_matches = inpatient[
    (inpatient["CLM_ID"].isin(segment_2_ids))
    &
    (inpatient["SEGMENT"] == 1)
]

print(
    "Segment 2 claims:",
    len(segment_2_ids)
)

print(
    "Matching segment 1 records:",
    len(segment_1_matches)
)

Segment 2 claims: 68
Matching segment 1 records: 68


In [22]:
print(
    segment_2[
        [
            "CLM_FROM_DT",
            "CLM_THRU_DT",
            "CLM_UTLZTN_DAY_CNT"
        ]
    ].isna().sum()
)

CLM_FROM_DT           68
CLM_THRU_DT           68
CLM_UTLZTN_DAY_CNT    68
dtype: int64


In [23]:
hcpcs_cols = [
    col for col in inpatient.columns
    if col.startswith("HCPCS_CD_")
]

inpatient = inpatient.drop(
    columns=hcpcs_cols
)

print("Removed HCPCS columns:", len(hcpcs_cols))
print("New shape:", inpatient.shape)

Removed HCPCS columns: 45
New shape: (66773, 37)


In [24]:
print(
    inpatient
    .groupby("SEGMENT")[
        "NCH_BENE_IP_DDCTBL_AMT"
    ]
    .apply(lambda x: x.isna().sum())
)

SEGMENT
1    2173
2       5
Name: NCH_BENE_IP_DDCTBL_AMT, dtype: int64


In [26]:
print(
    inpatient.groupby("SEGMENT")[
        [
            "CLM_FROM_DT",
            "CLM_THRU_DT",
            "CLM_UTLZTN_DAY_CNT",
            "NCH_BENE_IP_DDCTBL_AMT"
        ]
    ].apply(lambda x: x.isna().sum())
)

         CLM_FROM_DT  CLM_THRU_DT  CLM_UTLZTN_DAY_CNT  NCH_BENE_IP_DDCTBL_AMT
SEGMENT                                                                      
1                  0            0                   0                    2173
2                 68           68                  68                       5


In [27]:
print(
    "Negative CLM_PMT_AMT:",
    (inpatient["CLM_PMT_AMT"] < 0).sum()
)

Negative CLM_PMT_AMT: 55


In [28]:
inpatient["CLAIM_KEY"] = (
    inpatient["CLM_ID"].astype(str)
    + "_"
    + inpatient["SEGMENT"].astype(str)
)

print(
    "Rows:",
    len(inpatient)
)

print(
    "Unique CLAIM_KEY:",
    inpatient["CLAIM_KEY"].nunique()
)

Rows: 66773
Unique CLAIM_KEY: 66773


In [29]:
segment_2_ids = (
    inpatient.loc[
        inpatient["SEGMENT"] == 2,
        "CLM_ID"
    ]
)

paired = inpatient[
    inpatient["CLM_ID"].isin(segment_2_ids)
].sort_values(
    ["CLM_ID", "SEGMENT"]
)

print(
    paired[
        [
            "CLM_ID",
            "SEGMENT",
            "DESYNPUF_ID",
            "PRVDR_NUM",
            "CLM_PMT_AMT",
            "NCH_PRMRY_PYR_CLM_PD_AMT",
            "NCH_BENE_IP_DDCTBL_AMT",
            "CLM_UTLZTN_DAY_CNT"
        ]
    ].head(30).to_string(index=False)
)

         CLM_ID  SEGMENT      DESYNPUF_ID PRVDR_NUM  CLM_PMT_AMT  NCH_PRMRY_PYR_CLM_PD_AMT  NCH_BENE_IP_DDCTBL_AMT  CLM_UTLZTN_DAY_CNT
196011176996883        1 68ED7E3A8FEE7734    3602GC      34000.0                       0.0                  1068.0                 9.0
196011176996883        2 68ED7E3A8FEE7734    36016V       9000.0                       0.0                  1068.0                 NaN
196011177001033        1 A9877692C4640CB8    1001HB      23000.0                       0.0                  1068.0                12.0
196011177001033        2 A9877692C4640CB8    0506MK      17000.0                       0.0                  1068.0                 NaN
196021176981645        1 EAB1E19EA639510D    36008Q       5000.0                       0.0                  1024.0                 7.0
196021176981645        2 EAB1E19EA639510D    3600TU       5000.0                       0.0                  1024.0                 NaN
196031176958219        1 B440822AED0AF37E    2500QH    

In [30]:
segment_comparison = (
    paired
    .groupby("CLM_ID")
    .agg(
        beneficiary_count=("DESYNPUF_ID", "nunique"),
        provider_count=("PRVDR_NUM", "nunique"),
        segment_count=("SEGMENT", "count")
    )
)

print(segment_comparison.value_counts())

beneficiary_count  provider_count  segment_count
1                  2               2                48
                   1               2                20
Name: count, dtype: int64


In [31]:
print(
    paired[
        [
            "CLM_ID",
            "SEGMENT",
            "CLM_PMT_AMT",
            "NCH_PRMRY_PYR_CLM_PD_AMT"
        ]
    ].head(30).to_string(index=False)
)

         CLM_ID  SEGMENT  CLM_PMT_AMT  NCH_PRMRY_PYR_CLM_PD_AMT
196011176996883        1      34000.0                       0.0
196011176996883        2       9000.0                       0.0
196011177001033        1      23000.0                       0.0
196011177001033        2      17000.0                       0.0
196021176981645        1       5000.0                       0.0
196021176981645        2       5000.0                       0.0
196031176958219        1      10000.0                       0.0
196031176958219        2       8000.0                       0.0
196031176991132        1       7000.0                       0.0
196031176991132        2       9000.0                       0.0
196041177011944        1      27000.0                       0.0
196041177011944        2      23000.0                       0.0
196101177023665        1          0.0                       0.0
196101177023665        2      12000.0                       0.0
196121176967616        1       7000.0   

In [32]:
negative_payments = inpatient[
    inpatient["CLM_PMT_AMT"] < 0
]

print(
    negative_payments[
        [
            "CLM_ID",
            "SEGMENT",
            "DESYNPUF_ID",
            "PRVDR_NUM",
            "CLM_PMT_AMT",
            "NCH_PRMRY_PYR_CLM_PD_AMT"
        ]
    ].to_string(index=False)
)

print(
    negative_payments["SEGMENT"]
    .value_counts()
)

print(
    negative_payments["CLM_PMT_AMT"]
    .describe()
)

         CLM_ID  SEGMENT      DESYNPUF_ID PRVDR_NUM  CLM_PMT_AMT  NCH_PRMRY_PYR_CLM_PD_AMT
196811176982330        1 019E4729585EF3DD    2200TM       -100.0                    3000.0
196131177026233        1 05AE4A0DF82B9B48    1400DQ       -200.0                       0.0
196711176991366        1 08B6E69DF1A8719A    1001HB      -2000.0                       0.0
196091176978232        1 0D1F65FFA2BF36D3    1000NG       -200.0                       0.0
196561177019938        1 0D50A99E345BD92E    3600TD        -30.0                       0.0
196901176967000        1 0EA1D666FE43AA90    3200ZG       -300.0                       0.0
196471177022556        1 162E3D42199CB13A    5000MQ       -200.0                       0.0
196041177005785        1 17B7E6542074F88D    4900CN       -100.0                       0.0
196361177003626        1 19B6ECA664FCC497    3400XN      -3000.0                       0.0
196571176964136        1 1D99F4D0931E7CE6    14S2AA       -200.0                    4000.0

In [33]:
segment2_cols = [
    "CLM_ID",
    "SEGMENT",
    "ADMTNG_ICD9_DGNS_CD",
    "CLM_DRG_CD"
]

segment2_cols += [
    col for col in inpatient.columns
    if col.startswith("ICD9_DGNS_CD_")
]

segment2_cols += [
    col for col in inpatient.columns
    if col.startswith("ICD9_PRCDR_CD_")
]

print(
    inpatient[
        inpatient["SEGMENT"] == 2
    ][segment2_cols]
    .head(20)
    .to_string(index=False)
)

         CLM_ID  SEGMENT ADMTNG_ICD9_DGNS_CD CLM_DRG_CD ICD9_DGNS_CD_1 ICD9_DGNS_CD_2 ICD9_DGNS_CD_3 ICD9_DGNS_CD_4 ICD9_DGNS_CD_5 ICD9_DGNS_CD_6 ICD9_DGNS_CD_7 ICD9_DGNS_CD_8 ICD9_DGNS_CD_9 ICD9_DGNS_CD_10  ICD9_PRCDR_CD_1 ICD9_PRCDR_CD_2 ICD9_PRCDR_CD_3 ICD9_PRCDR_CD_4 ICD9_PRCDR_CD_5 ICD9_PRCDR_CD_6
196901176966106        2                 NaN        OTH            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN             NaN              NaN             NaN             NaN             NaN             NaN             NaN
196531176970382        2                 NaN        OTH            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN            NaN             NaN              NaN             NaN             NaN             NaN             NaN             NaN
196821176998887        2                 NaN        OTH            NaN            NaN   

In [36]:
inpatient["CLAIM_DURATION_DAYS"] = (
    inpatient["CLM_THRU_DT"]
    - inpatient["CLM_FROM_DT"]
).dt.days + 1
print(
    inpatient["CLAIM_DURATION_DAYS"].describe()
)
print(
    "Negative durations:",
    (
        inpatient["CLAIM_DURATION_DAYS"] < 0
    ).sum()
)

count    66705.000000
mean         6.677026
std          5.645399
min          1.000000
25%          3.000000
50%          5.000000
75%          8.000000
max         36.000000
Name: CLAIM_DURATION_DAYS, dtype: float64
Negative durations: 0


In [37]:
diagnosis_cols = [
    "ADMTNG_ICD9_DGNS_CD"
] + [
    f"ICD9_DGNS_CD_{i}"
    for i in range(1, 11)
]

inpatient["DIAGNOSIS_COUNT"] = (
    inpatient[diagnosis_cols]
    .notna()
    .sum(axis=1)
)

print(
    inpatient["DIAGNOSIS_COUNT"].describe()
)

count    66773.000000
mean         9.037261
std          1.884417
min          0.000000
25%          8.000000
50%         10.000000
75%         10.000000
max         11.000000
Name: DIAGNOSIS_COUNT, dtype: float64


In [38]:
procedure_cols = [
    f"ICD9_PRCDR_CD_{i}"
    for i in range(1, 7)
]

inpatient["PROCEDURE_COUNT"] = (
    inpatient[procedure_cols]
    .notna()
    .sum(axis=1)
)

print(
    inpatient["PROCEDURE_COUNT"].describe()
)

count    66773.000000
mean         1.435775
std          1.746302
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max          6.000000
Name: PROCEDURE_COUNT, dtype: float64


In [39]:
inpatient["HAS_NEGATIVE_PAYMENT"] = (
    inpatient["CLM_PMT_AMT"] < 0
).astype(int)
print(
    inpatient["HAS_NEGATIVE_PAYMENT"].value_counts()
)

HAS_NEGATIVE_PAYMENT
0    66718
1       55
Name: count, dtype: int64


In [40]:
inpatient["HAS_PRIMARY_PAYER_PAYMENT"] = (
    inpatient["NCH_PRMRY_PYR_CLM_PD_AMT"] > 0
).astype(int)
print(
    inpatient["HAS_PRIMARY_PAYER_PAYMENT"]
    .value_counts()
)

HAS_PRIMARY_PAYER_PAYMENT
0    65246
1     1527
Name: count, dtype: int64


In [41]:
provider_features = (
    inpatient
    .groupby("PRVDR_NUM")
    .agg(
        CLAIM_COUNT=("CLM_ID", "count"),
        UNIQUE_CLAIM_COUNT=("CLM_ID", "nunique"),
        TOTAL_PAYMENT=("CLM_PMT_AMT", "sum"),
        AVG_PAYMENT=("CLM_PMT_AMT", "mean"),
        MEDIAN_PAYMENT=("CLM_PMT_AMT", "median"),
        MAX_PAYMENT=("CLM_PMT_AMT", "max"),

        AVG_CLAIM_DURATION=("CLAIM_DURATION_DAYS", "mean"),
        MAX_CLAIM_DURATION=("CLAIM_DURATION_DAYS", "max"),

        AVG_DIAGNOSIS_COUNT=("DIAGNOSIS_COUNT", "mean"),
        AVG_PROCEDURE_COUNT=("PROCEDURE_COUNT", "mean"),

        NEGATIVE_PAYMENT_COUNT=("HAS_NEGATIVE_PAYMENT", "sum"),
        PRIMARY_PAYER_PAYMENT_COUNT=(
            "HAS_PRIMARY_PAYER_PAYMENT",
            "sum"
        )
    )
    .reset_index()
)
print(provider_features.shape)
print(provider_features.head())
print(
    provider_features.describe().T
)

(2675, 13)
  PRVDR_NUM  CLAIM_COUNT  UNIQUE_CLAIM_COUNT  TOTAL_PAYMENT   AVG_PAYMENT  \
0    01006H            9                   9       166000.0  18444.444444   
1    01006V           80                  80       727000.0   9087.500000   
2    0100AA            7                   7        83000.0  11857.142857   
3    0100AQ            1                   1         4000.0   4000.000000   
4    0100BB           14                  14       104000.0   7428.571429   

   MEDIAN_PAYMENT  MAX_PAYMENT  AVG_CLAIM_DURATION  MAX_CLAIM_DURATION  \
0         12000.0      42000.0            9.555556                28.0   
1          6000.0      57000.0            6.212500                22.0   
2         10000.0      21000.0            6.857143                14.0   
3          4000.0       4000.0            4.000000                 4.0   
4          5000.0      27000.0            6.071429                14.0   

   AVG_DIAGNOSIS_COUNT  AVG_PROCEDURE_COUNT  NEGATIVE_PAYMENT_COUNT  \
0         

In [42]:
provider_features["NEGATIVE_PAYMENT_RATE"] = (
    provider_features["NEGATIVE_PAYMENT_COUNT"]
    / provider_features["CLAIM_COUNT"]
)
provider_features["PRIMARY_PAYER_PAYMENT_RATE"] = (
    provider_features["PRIMARY_PAYER_PAYMENT_COUNT"]
    / provider_features["CLAIM_COUNT"]
)

In [43]:
print(
    provider_features.isna().sum()
    .sort_values(ascending=False)
)

PRVDR_NUM                      0
CLAIM_COUNT                    0
UNIQUE_CLAIM_COUNT             0
TOTAL_PAYMENT                  0
AVG_PAYMENT                    0
MEDIAN_PAYMENT                 0
MAX_PAYMENT                    0
AVG_CLAIM_DURATION             0
MAX_CLAIM_DURATION             0
AVG_DIAGNOSIS_COUNT            0
AVG_PROCEDURE_COUNT            0
NEGATIVE_PAYMENT_COUNT         0
PRIMARY_PAYER_PAYMENT_COUNT    0
NEGATIVE_PAYMENT_RATE          0
PRIMARY_PAYER_PAYMENT_RATE     0
dtype: int64


In [44]:
print(
    provider_features[
        [
            "CLAIM_COUNT",
            "TOTAL_PAYMENT",
            "AVG_PAYMENT",
            "MEDIAN_PAYMENT",
            "MAX_PAYMENT",
            "AVG_CLAIM_DURATION",
            "AVG_DIAGNOSIS_COUNT",
            "AVG_PROCEDURE_COUNT",
            "NEGATIVE_PAYMENT_RATE",
            "PRIMARY_PAYER_PAYMENT_RATE"
        ]
    ].describe().T
)

                             count           mean            std    min  \
CLAIM_COUNT                 2675.0      24.961869      47.859579    1.0   
TOTAL_PAYMENT               2675.0  238975.768224  463602.746971 -800.0   
AVG_PAYMENT                 2675.0    9553.876124    4152.846976 -800.0   
MEDIAN_PAYMENT              2675.0    7510.654206    3748.290378 -800.0   
MAX_PAYMENT                 2675.0   28999.813084   17742.482368 -800.0   
AVG_CLAIM_DURATION          2675.0       6.679129       2.512670    1.0   
AVG_DIAGNOSIS_COUNT         2675.0       9.048023       0.836667    2.0   
AVG_PROCEDURE_COUNT         2675.0       1.426535       0.783066    0.0   
NEGATIVE_PAYMENT_RATE       2675.0       0.001445       0.028568    0.0   
PRIMARY_PAYER_PAYMENT_RATE  2675.0       0.023611       0.067260    0.0   

                                     25%           50%            75%  \
CLAIM_COUNT                     4.000000     10.000000      24.000000   
TOTAL_PAYMENT               

In [45]:
beneficiary_concentration = (
    inpatient
    .groupby("PRVDR_NUM")
    .agg(
        UNIQUE_BENEFICIARIES=(
            "DESYNPUF_ID",
            "nunique"
        )
    )
    .reset_index()
)

provider_features = provider_features.merge(
    beneficiary_concentration,
    on="PRVDR_NUM",
    how="left"
)

In [46]:
provider_features["CLAIMS_PER_BENEFICIARY"] = (
    provider_features["CLAIM_COUNT"]
    / provider_features["UNIQUE_BENEFICIARIES"]
)

In [47]:
print(
    provider_features[
        [
            "CLAIM_COUNT",
            "UNIQUE_BENEFICIARIES",
            "CLAIMS_PER_BENEFICIARY"
        ]
    ].describe().T
)

                         count       mean        std  min  25%        50%  \
CLAIM_COUNT             2675.0  24.961869  47.859579  1.0  4.0  10.000000   
UNIQUE_BENEFICIARIES    2675.0  20.310280  36.539967  1.0  4.0   9.000000   
CLAIMS_PER_BENEFICIARY  2675.0   1.151879   0.247488  1.0  1.0   1.041667   

                              75%    max  
CLAIM_COUNT             24.000000  772.0  
UNIQUE_BENEFICIARIES    21.000000  580.0  
CLAIMS_PER_BENEFICIARY   1.223356    4.0  


In [48]:
payment_variability = (
    inpatient
    .groupby("PRVDR_NUM")
    .agg(
        PAYMENT_STD=("CLM_PMT_AMT", "std")
    )
    .reset_index()
)

provider_features = provider_features.merge(
    payment_variability,
    on="PRVDR_NUM",
    how="left"
)

In [49]:
print(
    provider_features["PAYMENT_STD"].describe()
)

count     2466.000000
mean      8076.473490
std       4690.569750
min          0.000000
25%       4599.561358
50%       7557.572358
75%      10385.584692
max      37476.659403
Name: PAYMENT_STD, dtype: float64


In [50]:
provider_features["PAYMENT_STD"] = (
    provider_features["PAYMENT_STD"]
    .fillna(0)
)
duration_variability = (
    inpatient
    .groupby("PRVDR_NUM")
    .agg(
        CLAIM_DURATION_STD=(
            "CLAIM_DURATION_DAYS",
            "std"
        )
    )
    .reset_index()
)

provider_features = provider_features.merge(
    duration_variability,
    on="PRVDR_NUM",
    how="left"
)

provider_features["CLAIM_DURATION_STD"] = (
    provider_features["CLAIM_DURATION_STD"]
    .fillna(0)
)

In [51]:
print(provider_features.shape)
print(provider_features.columns.tolist())
print(
    provider_features.describe().T
)
print(
    provider_features.isna().sum()
)

(2675, 19)
['PRVDR_NUM', 'CLAIM_COUNT', 'UNIQUE_CLAIM_COUNT', 'TOTAL_PAYMENT', 'AVG_PAYMENT', 'MEDIAN_PAYMENT', 'MAX_PAYMENT', 'AVG_CLAIM_DURATION', 'MAX_CLAIM_DURATION', 'AVG_DIAGNOSIS_COUNT', 'AVG_PROCEDURE_COUNT', 'NEGATIVE_PAYMENT_COUNT', 'PRIMARY_PAYER_PAYMENT_COUNT', 'NEGATIVE_PAYMENT_RATE', 'PRIMARY_PAYER_PAYMENT_RATE', 'UNIQUE_BENEFICIARIES', 'CLAIMS_PER_BENEFICIARY', 'PAYMENT_STD', 'CLAIM_DURATION_STD']
                              count           mean            std    min  \
CLAIM_COUNT                  2675.0      24.961869      47.859579    1.0   
UNIQUE_CLAIM_COUNT           2675.0      24.954393      47.837173    1.0   
TOTAL_PAYMENT                2675.0  238975.768224  463602.746971 -800.0   
AVG_PAYMENT                  2675.0    9553.876124    4152.846976 -800.0   
MEDIAN_PAYMENT               2675.0    7510.654206    3748.290378 -800.0   
MAX_PAYMENT                  2675.0   28999.813084   17742.482368 -800.0   
AVG_CLAIM_DURATION           2675.0       6.679129  

In [ ]:
inpatient["HAS_PRIMARY_PAYER_PAYMENT"] = (
    inpatient["NCH_PRMRY_PYR_CLM_PD_AMT"] > 0
).astype(int)
print(
    inpatient["HAS_PRIMARY_PAYER_PAYMENT"]
    .value_counts()
)

HAS_PRIMARY_PAYER_PAYMENT
0    65246
1     1527
Name: count, dtype: int64


In [52]:
cleaned_inpatient = inpatient.copy()

cleaned_inpatient.to_csv(
    "inpatient_claims_cleaned.csv",
    index=False
)

print("Saved:", "inpatient_claims_cleaned.csv")
print("Shape:", cleaned_inpatient.shape)

Saved: inpatient_claims_cleaned.csv
Shape: (66773, 42)


In [53]:
provider_features.to_csv(
    "inpatient_provider_features.csv",
    index=False
)

print("Saved:", "inpatient_provider_features.csv")
print("Shape:", provider_features.shape)

Saved: inpatient_provider_features.csv
Shape: (2675, 19)


In [54]:
import os

print(os.path.exists("inpatient_claims_cleaned.csv"))
print(os.path.exists("inpatient_provider_features.csv"))

True
True


In [3]:
import pandas as pd
from pathlib import Path

file = Path(
    "../data/processed/primary/"
    "inpatient_provider_features.csv"
)

inpatient = pd.read_csv(file)

print("Shape:", inpatient.shape)
print(inpatient.head())
print(inpatient.columns.tolist())

Shape: (2675, 19)
  PRVDR_NUM  CLAIM_COUNT  UNIQUE_CLAIM_COUNT  TOTAL_PAYMENT   AVG_PAYMENT  \
0    01006H            9                   9       166000.0  18444.444444   
1    01006V           80                  80       727000.0   9087.500000   
2    0100AA            7                   7        83000.0  11857.142857   
3    0100AQ            1                   1         4000.0   4000.000000   
4    0100BB           14                  14       104000.0   7428.571429   

   MEDIAN_PAYMENT  MAX_PAYMENT  AVG_CLAIM_DURATION  MAX_CLAIM_DURATION  \
0         12000.0      42000.0            9.555556                28.0   
1          6000.0      57000.0            6.212500                22.0   
2         10000.0      21000.0            6.857143                14.0   
3          4000.0       4000.0            4.000000                 4.0   
4          5000.0      27000.0            6.071429                14.0   

   AVG_DIAGNOSIS_COUNT  AVG_PROCEDURE_COUNT  NEGATIVE_PAYMENT_COUNT  \
0  